In [1]:
import os, time, json, httpx
from dotenv import load_dotenv

load_dotenv()
LLM = httpx.Client(base_url="https://api.cerebras.ai/v1", timeout=30,
                   headers={"Authorization": f"Bearer {os.environ['CEREBUS']}"})
MODEL = "gpt-oss-120b"

def timeit(prompt, system="Respond shortly.", stream=True):
    body = {"model": MODEL, "stream": stream,
            "messages": [{"role": "system", "content": system},
                         {"role": "user", "content": prompt}]}
    t0 = time.perf_counter()
    if not stream:
        r = LLM.post("/chat/completions", json=body).json()
        total = time.perf_counter() - t0
        text = r["choices"][0]["message"]["content"]
        print(f"{text}\n\ntotal {total:.3f}s")
        return total
    first, text = None, ""
    with LLM.stream("POST", "/chat/completions", json=body) as r:
        for line in r.iter_lines():
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            d = json.loads(line[6:])["choices"][0]["delta"].get("content")
            if d:
                first = first or time.perf_counter() - t0   # time to first token
                text += d
    total = time.perf_counter() - t0
    print(f"{text}\n\nfirst token {first:.3f}s | total {total:.3f}s")
    return first, total

In [2]:
PROMPT = "How far away is the moon?"

timeit(PROMPT)

The Moon is about **384,400 km** (≈ 238,855 mi) from Earth on average—ranging from ~363,300 km at its closest (perigee) to ~405,500 km at its farthest (apogee).

first token 0.329s | total 0.355s


(0.32880779216066003, 0.35467291716486216)

In [3]:
# same prompt, N times - warm connection, median is the honest number
import statistics
runs = [timeit(PROMPT)[0] for _ in range(5)]
print(f"\nmedian first token {statistics.median(runs):.3f}s")

The Moon is about 384,400 km (≈ 238,900 mi) from Earth on average, though its distance varies between roughly 363,000 km and 405,000 km due to its elliptical orbit.

first token 0.336s | total 0.361s
About 384,400 kilometers (≈239,000 miles) on average.

first token 0.430s | total 0.436s
The Moon is about **384 400 km** (≈ 238 855 mi) from Earth on average, though its distance varies between roughly 363 000 km and 405 000 km due to its elliptical orbit.

first token 0.309s | total 0.329s
The Moon is about **384,400 km (≈ 238,900 mi)** from Earth on average, ranging roughly from 363,300 km at perigee to 405,500 km at apogee.

first token 0.311s | total 0.334s
The Moon is about 384,400 km (≈ 238,855 mi) from Earth on average, though the distance varies between roughly 363,300 km and 405,500 km due to its elliptical orbit.

first token 0.280s | total 0.304s

median first token 0.311s
